# Crop Price Forecasting API for GreenField Farmers
## Task 1: Data Preparation, Cleaning, and Quality Validation

**Author:** Mushtaq Ahmad  
**Client:** GreenField Farmers Co-op  
**Objective:** Build and validate the data pipeline for GreenField Farmers Co-op's top 3 staple crops (**Wheat**, **Rice**, **Potato**).

---

### 1. Environment & Library Ingestion

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root in sys.path
sys.path.append(os.path.abspath('..'))
from src.data_pipeline import CropDataPipeline
from src.data_validator import CropDataValidator

### 2. Inspecting Raw Public Agricultural Dataset
Let us examine the raw dataset containing artifacts (inconsistent date representations, missing prices/arrivals).

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'crop_prices_raw.csv')
df_raw = pd.read_csv(raw_path)

print(f"Raw dataset dimensions: {df_raw.shape}")
print("\n--- Null Value Count per Column ---")
print(df_raw.isna().sum())
df_raw.head(8)

### 3. Date Format Heterogeneity Analysis
Notice the heterogeneous date strings (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`) in raw market reporting logs.

In [ ]:
sample_dates = df_raw['Date'].head(10).tolist()
print("Sample raw date entries:", sample_dates)

### 4. Executing the Automated Cleaning Pipeline
Running `CropDataPipeline` to standardize dates into ISO-8601 (`YYYY-MM-DD`), interpolate missing price series, and impute arrival volumes.

In [ ]:
pipeline = CropDataPipeline(
    raw_data_path=raw_path,
    processed_data_path=os.path.join('..', 'data', 'processed', 'crop_prices_cleaned.csv')
)
df_cleaned = pipeline.run_pipeline()

print("\nCleaned DataFrame Preview:")
df_cleaned.head(6)

### 5. Automated Data Quality & Validation Checks
Running all 5 automated data validation rules via `CropDataValidator`.

In [ ]:
validator = CropDataValidator(processed_data_path=os.path.join('..', 'data', 'processed', 'crop_prices_cleaned.csv'))
validation_status = validator.run_all_checks()

### 6. Exploratory Visualizations: Top 3 Crop Price Trajectories

In [ ]:
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'])
plt.figure(figsize=(12, 6))

for crop in ['Wheat', 'Rice', 'Potato']:
    crop_subset = df_cleaned[df_cleaned['Crop'] == crop]
    plt.plot(crop_subset['Date'], crop_subset['Modal_Price_PKR'], label=crop, linewidth=2)

plt.title('GreenField Farmers Co-op: Cleaned Historical Price Trends (2022-2024)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Wholesale Modal Price (PKR/Quintal)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()